In [ ]:
%load_ext autoreload
%autoreload 3
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import sys, os, pickle, warnings, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import fdrcorrection

sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/AP_empirical_paper1/datasets/spe-1/spe1_helper_modules/')
from config import (SPE1_PICKLE_ROOT, DICT_CELL_TYPE, DICT_PATCH_TYPE,
                    DICT_CORT_DEPTH, DICT_DARK_NEURONS, DICT_EAP_WAV)
from pop_ridge_utils import load_population_results
from ridge_regression_utils import WAVEFORM_LABELS, FEAT_LABELS

warnings.filterwarnings('ignore')

_CB = ['#0072B2', '#D55E00', '#009E73', '#CC79A7', '#56B4E9', '#E69F00']
_SIG_COL   = '#D55E00'
_INSIG_COL = '#56B4E9'

In [ ]:
# ── Load ridge regression results ─────────────────────────────────────────────
RIDGE_PICKLE_DIR = os.path.join(SPE1_PICKLE_ROOT, 'ridge_regression_pickles')

all_results, cell_ids, target_names, predictor_sets = load_population_results(RIDGE_PICKLE_DIR)

feat_labels   = FEAT_LABELS
target_labels = (
    [f'Pre {l}'    for l in feat_labels] +
    [f'Pre-BL {l}' for l in feat_labels] +
    [f'Delta {l}'  for l in feat_labels]
)

# Load population summary pickle
pop_path = os.path.join(RIDGE_PICKLE_DIR, 'population_ridge_results.pkl')
with open(pop_path, 'rb') as f:
    pop = pickle.load(f)

r2_pop        = pop['r2_pop']
sig_pop       = pop['sig_pop']
beta_pop      = pop['beta_pop']
df_tests      = pop['df_tests']
mean_beta_mat = pop['mean_beta_mat']
sig_beta_mat  = pop['sig_beta_mat']

print(f'{len(cell_ids)} cells  |  {len(target_names)} targets')

In [ ]:
# ── Build per-cell metadata + ridge stats DataFrame ───────────────────────────
rows = []
for cid in cell_ids:
    cell_num = int(cid.replace('c', ''))
    res = all_results[cid]

    row = dict(
        cell_id      = cid,
        cell_num     = cell_num,
        cell_type    = DICT_CELL_TYPE.get(cell_num, 'unknown'),
        patch_type   = DICT_PATCH_TYPE.get(cell_num, 'unknown'),
        cort_depth   = DICT_CORT_DEPTH.get(cell_num, np.nan),
        dark_neuron  = str(DICT_DARK_NEURONS.get(cell_num, np.nan)),
        eap_visible  = str(DICT_EAP_WAV.get(cell_num, np.nan)),
        n_spikes     = res[target_names[0]]['Waveform only'].get('n_valid', np.nan),
        n_sig_wv     = sum(res[tn]['Waveform only'].get('sig_fdr', False) for tn in target_names),
        n_sig_isi    = sum(res[tn]['Log ISI only'].get('sig_fdr', False) for tn in target_names),
        n_sig_both   = sum(res[tn]['Waveform + Log ISI'].get('sig_fdr', False) for tn in target_names),
        mean_r2_wv   = float(np.nanmean([res[tn]['Waveform only'].get('r2_cv', np.nan) for tn in target_names])),
    )
    for tn, tl in zip(target_names, target_labels):
        row[f'r2_{tn}']    = res[tn]['Waveform only'].get('r2_cv', np.nan)
        row[f'sig_{tn}']   = res[tn]['Waveform only'].get('sig_fdr', False)
        row[f'alpha_{tn}'] = res[tn]['Waveform only'].get('best_alpha', np.nan)
    rows.append(row)

df = pd.DataFrame(rows)

# Replace 'nan' strings with actual NaN so they're excluded from plots
for col in ['dark_neuron', 'eap_visible']:
    df[col] = df[col].replace('nan', np.nan)

print(df[['cell_id','cell_type','dark_neuron','eap_visible','n_spikes','n_sig_wv','mean_r2_wv']].to_string(index=False))

In [ ]:
# ── Cell × target significance heatmap ────────────────────────────────────────
# Sorted by total n significant targets, grouped by target type
n_sig_per_cell = sig_matrix.sum(axis=1)
sort_idx = np.argsort(n_sig_per_cell)[::-1]
cell_ids_sorted = [cell_ids[i] for i in sort_idx]

n_cells = len(cell_ids)
fig, ax = plt.subplots(figsize=(13, max(5, 0.32 * n_cells + 2)))

im = ax.imshow(sig_matrix[sort_idx], aspect='auto', cmap='Blues', vmin=0, vmax=1)

# Column group dividers and labels
for d in [4.5, 9.5]:
    ax.axvline(d, color='white', lw=2.5)
for x, lbl in [(2, 'Pre (abs)'), (7, 'Pre − BL'), (12, 'Δ post−pre')]:
    ax.text(x, -1.2, lbl, ha='center', va='top', fontsize=9,
            fontweight='bold', color='#444', transform=ax.get_xaxis_transform())

ax.set_xticks(range(len(target_names)))
ax.set_xticklabels([tl.split(' ', 1)[-1] for tl in target_labels],
                   rotation=40, ha='right', fontsize=8)
ax.set_yticks(range(n_cells))
yticklabels = [f'{cell_ids_sorted[i]}  ({int(n_sig_per_cell[sort_idx[i]])})'
               for i in range(n_cells)]
ax.set_yticklabels(yticklabels, fontsize=7.5)

ax.set_xlabel('LFP target', fontsize=10)
ax.set_ylabel('Cell  (n significant targets)', fontsize=10)
ax.set_title('Which cells have FDR-significant Waveform-only models?\n'
             '(sorted by total n significant, value = fraction per group)',
             fontsize=11, pad=14)

cbar = plt.colorbar(im, ax=ax, shrink=0.4, pad=0.01)
cbar.set_ticks([0, 1]); cbar.set_ticklabels(['Not sig.', 'Sig.'])

# Fraction annotation per target column
for c in range(len(target_names)):
    frac = sig_matrix[:, c].mean()
    ax.text(c, n_cells - 0.05, f'{frac:.0%}', ha='center', va='bottom',
            fontsize=6.5, color='#555', transform=ax.transData)

fig.tight_layout()
plt.show()

print(f'\nCells with ≥1 significant target: {(n_sig_per_cell > 0).sum()}/{n_cells}')
print(f'Targets with ≥1 significant cell: {(sig_matrix.sum(axis=0) > 0).sum()}/{len(target_names)}')

In [ ]:
# ── Distribution of CV R² per target ──────────────────────────────────────────
r2_data = [[df[f'r2_{tn}'].dropna().values for tn in target_names]]

fig, ax = plt.subplots(figsize=(14, 4))
positions = np.arange(len(target_names))

for t_idx, tn in enumerate(target_names):
    vals = df[f'r2_{tn}'].dropna().values
    bp = ax.boxplot(vals, positions=[t_idx], widths=0.5, patch_artist=True,
                    medianprops=dict(color='k', lw=2),
                    boxprops=dict(facecolor=_CB[0], alpha=0.5),
                    whiskerprops=dict(color='k'), capprops=dict(color='k'),
                    flierprops=dict(marker='o', markersize=3, alpha=0.4))
    ax.scatter(np.random.default_rng(t_idx).uniform(t_idx - 0.2, t_idx + 0.2, len(vals)),
               vals, alpha=0.5, s=15, color=_CB[0], zorder=3)

ax.axhline(0, color='gray', lw=1, ls='--')
for d in [4.5, 9.5]: ax.axvline(d, color='gray', lw=1.2, ls='--')
ax.set_xticks(positions)
ax.set_xticklabels(target_labels, rotation=35, ha='right', fontsize=8)
ax.set_ylabel('5-fold CV R²', fontsize=10)
ax.set_title('Distribution of CV R² across cells (Waveform only)', fontsize=11)
sns.despine(ax=ax)
fig.tight_layout()
plt.show()

In [ ]:
# ── Best alpha distribution per target (log scale) ────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))

for t_idx, tn in enumerate(target_names):
    vals = np.log10(df[f'alpha_{tn}'].dropna().values.clip(1e-4))
    ax.boxplot(vals, positions=[t_idx], widths=0.5, patch_artist=True,
               medianprops=dict(color='k', lw=2),
               boxprops=dict(facecolor=_CB[2], alpha=0.5),
               whiskerprops=dict(color='k'), capprops=dict(color='k'),
               flierprops=dict(marker='o', markersize=3, alpha=0.4))

for d in [4.5, 9.5]: ax.axvline(d, color='gray', lw=1.2, ls='--')
ax.set_xticks(range(len(target_names)))
ax.set_xticklabels(target_labels, rotation=35, ha='right', fontsize=8)
ax.set_ylabel('log₁₀(best alpha)', fontsize=10)
ax.set_title('Distribution of selected Ridge alpha per target (Waveform only)', fontsize=11)
sns.despine(ax=ax)
fig.tight_layout()
plt.show()

In [ ]:
# ── Does metadata predict mean CV R²? ─────────────────────────────────────────
# Kruskal-Wallis for categorical vars, Spearman for cortical depth.

cat_vars = {
    'Cell type':       'cell_type',
    'Patch type':      'patch_type',
    'Dark neuron':     'dark_neuron',
    'EAP visible':     'eap_visible',
}

fig, axes = plt.subplots(1, len(cat_vars), figsize=(4 * len(cat_vars), 4), sharey=True)
fig.suptitle('Metadata × mean CV R² (Waveform only)', fontsize=11)

for ax, (label, col) in zip(axes, cat_vars.items()):
    d = df[['mean_r2_wv', col]].dropna()
    order = sorted(d[col].unique())
    if len(order) < 2:
        ax.set_visible(False); continue
    colors = [_CB[i % len(_CB)] for i in range(len(order))]
    sns.boxplot(data=d, x=col, y='mean_r2_wv', order=order,
                palette=dict(zip(order, colors)), width=0.5, ax=ax,
                fliersize=0, boxprops={'alpha': 0.5})
    sns.stripplot(data=d, x=col, y='mean_r2_wv', order=order,
                  palette=dict(zip(order, colors)), alpha=0.7, jitter=0.15, size=6, ax=ax)
    ax.axhline(0, color='gray', lw=0.8, ls='--')
    groups = [d[d[col] == g]['mean_r2_wv'].values for g in order]
    if all(len(g) >= 3 for g in groups):
        _, p = stats.kruskal(*groups)
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        color = _SIG_COL if sig != 'ns' else '#888'
        ax.text(0.5, 1.02, sig, transform=ax.transAxes, ha='center', fontsize=12,
                fontweight='bold', color=color)
    ax.set_title(label, fontsize=9)
    ax.set_xlabel('')
    if ax == axes[0]: ax.set_ylabel('Mean CV R²', fontsize=9)
    sns.despine(ax=ax)

fig.tight_layout()
plt.show()

# Spearman: cortical depth
d_depth = df[['mean_r2_wv', 'cort_depth']].dropna()
if len(d_depth) >= 5:
    rho, p = stats.spearmanr(d_depth['cort_depth'], d_depth['mean_r2_wv'])
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.scatter(d_depth['cort_depth'], d_depth['mean_r2_wv'], alpha=0.7, color=_CB[0])
    ax.set_xlabel('Cortical depth (µm)', fontsize=9)
    ax.set_ylabel('Mean CV R²', fontsize=9)
    ax.set_title(f'Cortical depth × mean R²\nSpearman ρ={rho:.3f}, p={p:.3f}', fontsize=9)
    sns.despine(ax=ax)
    plt.tight_layout(); plt.show()

In [ ]:
# ── Does n_spikes predict R²? (sample size confound check) ───────────────────
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(df['n_spikes'], df['mean_r2_wv'], alpha=0.7, color=_CB[0])
rho, p = stats.spearmanr(df['n_spikes'].dropna(), df['mean_r2_wv'].dropna())
ax.set_xlabel('N spikes', fontsize=9)
ax.set_ylabel('Mean CV R² (Waveform only)', fontsize=9)
ax.set_title(f'Sample size vs model performance\nSpearman ρ={rho:.3f}, p={p:.3f}', fontsize=9)
sns.despine(ax=ax)
plt.tight_layout()
plt.show()
print('If ρ is large and significant, bigger cells drive R² — a sample size confound.')
print('If ns, R² reflects a real relationship not inflated by having more spikes.')

In [ ]:
# ── Does metadata predict beta weight direction for FDR-significant pop pairs? ─
sig_pairs = df_tests[df_tests.sig_fdr][['target', 'target_label', 'feature']].values.tolist()

if not sig_pairs:
    print('No population-level FDR-significant (target, feature) pairs to analyse.')
else:
    for tn, tl, feat in sig_pairs:
        beta_vals = np.array(beta_pop[tn][feat])
        df_b = df[['cell_id', 'cell_num', 'cell_type', 'patch_type',
                   'dark_neuron', 'eap_visible']].copy()
        df_b['beta'] = beta_vals
        df_b = df_b.dropna(subset=['beta'])

        print(f'\n{tl} × {feat}')
        ncols = len(cat_vars)
        fig, axes = plt.subplots(1, ncols, figsize=(4 * ncols, 3.5), sharey=True)
        fig.suptitle(f'{tl} × {feat}  — Beta direction by metadata', fontsize=10)

        for ax, (label, col) in zip(axes, cat_vars.items()):
            d = df_b[['beta', col]].dropna()
            order = sorted(d[col].unique())
            if len(order) < 2:
                ax.set_visible(False); continue
            colors = [_CB[i % len(_CB)] for i in range(len(order))]
            sns.boxplot(data=d, x=col, y='beta', order=order,
                        palette=dict(zip(order, colors)), width=0.5, ax=ax,
                        fliersize=0, boxprops={'alpha': 0.5})
            sns.stripplot(data=d, x=col, y='beta', order=order,
                          palette=dict(zip(order, colors)),
                          alpha=0.7, jitter=0.15, size=6, ax=ax)
            ax.axhline(0, color='gray', lw=0.8, ls='--')
            groups = [d[d[col] == g]['beta'].values for g in order]
            if all(len(g) >= 3 for g in groups):
                _, p = stats.kruskal(*groups)
                sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
                color = _SIG_COL if sig != 'ns' else '#888'
                ax.text(0.5, 1.02, sig, transform=ax.transAxes, ha='center',
                        fontsize=12, fontweight='bold', color=color)
            ax.set_title(label, fontsize=9)
            ax.set_xlabel('')
            if ax == axes[0]: ax.set_ylabel('β (std units)', fontsize=9)
            sns.despine(ax=ax)

        fig.tight_layout()
        plt.show()

In [ ]:
# ── Number of significant targets per cell ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Histogram
axes[0].hist(df['n_sig_wv'], bins=range(0, len(target_names) + 2),
             color=_CB[0], alpha=0.8, edgecolor='white')
axes[0].set_xlabel('N significant targets per cell (Waveform only)', fontsize=9)
axes[0].set_ylabel('N cells', fontsize=9)
axes[0].set_title('How many targets does each cell predict?', fontsize=10)
sns.despine(ax=axes[0])

# Breakdown per predictor set
x = np.arange(3)
means = [df['n_sig_wv'].mean(), df['n_sig_isi'].mean(), df['n_sig_both'].mean()]
sems  = [stats.sem(df['n_sig_wv']), stats.sem(df['n_sig_isi']), stats.sem(df['n_sig_both'])]
axes[1].bar(x, means, yerr=sems, color=_CB[:3], alpha=0.8, capsize=5)
axes[1].set_xticks(x)
axes[1].set_xticklabels(['Waveform only', 'Log ISI only', 'Waveform + ISI'], rotation=15, ha='right', fontsize=9)
axes[1].set_ylabel('Mean N significant targets per cell', fontsize=9)
axes[1].set_title('Predictor set comparison', fontsize=10)
sns.despine(ax=axes[1])

fig.tight_layout()
plt.show()